# 3.1 Stratégies de Fusion - Stratégie 1

Combinaison des embeddings textuels et visuels par concaténation simple.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

In [2]:
# Charger les données
df = pd.read_csv('data/subset_fashion_dataset/products_final.csv')
ground_truth = pd.read_csv('data/subset_fashion_dataset/ground_truth.csv', sep=';')

# Charger les embeddings existants
text_embeddings = np.load('results/text_embeddings.npy')  # MiniLM: 384 dim
vit_embeddings = np.load('results/vit_embeddings.npy')    # ViT: 768 dim

print(f"Produits: {len(df)}")
print(f"Text embeddings: {text_embeddings.shape}")
print(f"ViT embeddings: {vit_embeddings.shape}")

Produits: 400
Text embeddings: (400, 384)
ViT embeddings: (400, 768)


## Stratégie 1 : Concaténation Simple

**Formule** : `e_multimodal = concatenate(e_text_L2, e_image_L2)`

- Normalisation L2 de chaque modalité
- Concaténation des vecteurs
- Dimension finale : 384 + 768 = 1152

In [3]:
# Normalisation L2
text_norm = normalize(text_embeddings, norm='l2')
vit_norm = normalize(vit_embeddings, norm='l2')

# Concaténation
multimodal_embeddings = np.concatenate([text_norm, vit_norm], axis=1)

print(f"Multimodal embeddings: {multimodal_embeddings.shape}")
print(f"Norme d'un vecteur texte: {np.linalg.norm(text_norm[0]):.4f}")
print(f"Norme d'un vecteur ViT: {np.linalg.norm(vit_norm[0]):.4f}")
print(f"Norme d'un vecteur multimodal: {np.linalg.norm(multimodal_embeddings[0]):.4f}")

Multimodal embeddings: (400, 1152)
Norme d'un vecteur texte: 1.0000
Norme d'un vecteur ViT: 1.0000
Norme d'un vecteur multimodal: 1.4142


In [4]:
# Matrice de similarité cosinus
multimodal_similarity = cosine_similarity(multimodal_embeddings)

print(f"Matrice de similarité: {multimodal_similarity.shape}")
print(f"Similarité max: {multimodal_similarity.max():.4f}")
print(f"Similarité min (hors diagonale): {np.sort(multimodal_similarity.flatten())[:-len(df)][-1]:.4f}")

Matrice de similarité: (400, 400)
Similarité max: 1.0000
Similarité min (hors diagonale): 0.9387


In [5]:
# Extraire top-5 pour les ancres
anchor_ids = ground_truth['anchor_id'].values
results = []

for anchor_id in anchor_ids:
    anchor_idx = df[df['id'] == anchor_id].index[0]
    
    # Récupérer les similarités pour cette ancre
    similarities = multimodal_similarity[anchor_idx]
    
    # Top-5 (exclure soi-même)
    similarities[anchor_idx] = -1
    top5_indices = np.argsort(similarities)[-5:][::-1]
    top5_ids = df.iloc[top5_indices]['id'].values
    top5_similarities = similarities[top5_indices]
    
    results.append({
        'anchor_id': anchor_id,
        'top5_ids': list(top5_ids),
        'top5_similarities': list(top5_similarities)
    })

df_strategy1 = pd.DataFrame(results)
print(f"Top-5 extraits pour {len(df_strategy1)} ancres")
print(f"\nExemple ancre {df_strategy1['anchor_id'].iloc[0]}:")
print(f"  IDs: {df_strategy1['top5_ids'].iloc[0]}")
print(f"  Similarités: {[f'{s:.4f}' for s in df_strategy1['top5_similarities'].iloc[0]]}")

Top-5 extraits pour 30 ancres

Exemple ancre 3314:
  IDs: [np.int64(26486), np.int64(31279), np.int64(39345), np.int64(8389), np.int64(4861)]
  Similarités: ['0.6494', '0.6101', '0.6099', '0.6034', '0.5990']


In [6]:
# Sauvegarder
df_strategy1.to_csv('results/strategy1_concatenation_top5.csv', index=False)
np.save('results/strategy1_multimodal_embeddings.npy', multimodal_embeddings)
np.save('results/strategy1_similarity_matrix.npy', multimodal_similarity)

print("Sauvegardé:")
print("  - results/strategy1_concatenation_top5.csv")
print("  - results/strategy1_multimodal_embeddings.npy")
print("  - results/strategy1_similarity_matrix.npy")

Sauvegardé:
  - results/strategy1_concatenation_top5.csv
  - results/strategy1_multimodal_embeddings.npy
  - results/strategy1_similarity_matrix.npy


In [7]:
# Évaluation
def evaluate_method(predictions_df, ground_truth_df):
    precisions = []
    recalls = []
    mrrs = []
    
    for _, pred_row in predictions_df.iterrows():
        anchor_id = pred_row['anchor_id']
        pred_ids = set(pred_row['top5_ids'])
        
        gt_row = ground_truth_df[ground_truth_df['anchor_id'] == anchor_id]
        if len(gt_row) == 0:
            continue
        gt_row = gt_row.iloc[0]
        
        true_ids = set()
        for i in range(1, 6):
            gid = gt_row[f'global_id{i}']
            if pd.notna(gid):
                true_ids.add(int(gid))
        
        intersection = pred_ids & true_ids
        precisions.append(len(intersection) / 5)
        recalls.append(len(intersection) / 5)
        
        pred_list = list(pred_row['top5_ids'])
        mrr = 0.0
        for rank, pid in enumerate(pred_list, 1):
            if pid in true_ids:
                mrr = 1.0 / rank
                break
        mrrs.append(mrr)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'mrr': np.mean(mrrs)
}

metrics_s1 = evaluate_method(df_strategy1, ground_truth)
print(f"Stratégie 1 (Concaténation): P@5={metrics_s1['precision']:.3f}, R@5={metrics_s1['recall']:.3f}, MRR={metrics_s1['mrr']:.3f}")

Stratégie 1 (Concaténation): P@5=0.533, R@5=0.533, MRR=0.819


In [8]:
# Comparaison avec méthodes unimodales
df_unimodal = pd.read_csv('results/evaluation_summary.csv')

df_comparison = pd.DataFrame([
    {'Méthode': 'TF-IDF', **df_unimodal.iloc[0][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'Embeddings', **df_unimodal.iloc[1][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'ResNet-50', **df_unimodal.iloc[2][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'ViT', **df_unimodal.iloc[3][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'Stratégie 1 (Concaténation)', **metrics_s1}
])

print("\n" + "="*80)
print("COMPARAISON UNIMODAL vs MULTIMODAL")
print("="*80)
print(f"{'Méthode':<30} | {'Precision@5':>12} | {'Recall@5':>12} | {'MRR':>12}")
print("-"*80)
for _, row in df_comparison.iterrows():
    p = f"{row['precision']*100:.1f}%"
    r = f"{row['recall']*100:.1f}%"
    m = f"{row['mrr']:.3f}"
    marker = " ★" if row['Méthode'] == 'Stratégie 1 (Concaténation)' else ""
    print(f"{row['Méthode']:<30} | {p:>12} | {r:>12} | {m:>12}{marker}")
print("="*80)


COMPARAISON UNIMODAL vs MULTIMODAL
Méthode                        |  Precision@5 |     Recall@5 |          MRR
--------------------------------------------------------------------------------
TF-IDF                         |        36.0% |        36.0% |        0.662
Embeddings                     |        36.7% |        36.7% |        0.661
ResNet-50                      |        38.7% |        38.7% |        0.364
ViT                            |        44.7% |        44.7% |        0.316
Stratégie 1 (Concaténation)    |        53.3% |        53.3% |        0.819 ★


In [9]:
# Sauvegarder la comparaison
df_comparison.to_csv('results/strategy1_comparison.csv', index=False)
print("\nSauvegardé: results/strategy1_comparison.csv")


Sauvegardé: results/strategy1_comparison.csv


## Stratégie 2 : Moyenne Pondérée

**Formule** : `e_multimodal = α × e_text_proj + (1-α) × e_image_proj`

- Problème : dimensions différentes (384 vs 768)
- Solution : PCA vers dimension commune (512)
- Tester α ∈ {0.3, 0.5, 0.7} pour trouver le meilleur poids

In [10]:
from sklearn.decomposition import PCA

# Normalisation L2
text_norm = normalize(text_embeddings, norm='l2')
vit_norm = normalize(vit_embeddings, norm='l2')

# Approche 1 : projeter seulement l'image vers 384
text_proj = text_norm  # Pas de projection pour le texte
pca_vit = PCA(n_components=384)
vit_proj = pca_vit.fit_transform(vit_norm)  # 768 → 384

print(f"Texte: {text_proj.shape}")
print(f"ViT projeté: {vit_proj.shape}")
print(f"Variance expliquée ViT: {pca_vit.explained_variance_ratio_.sum():.3f}")


Texte: (400, 384)
ViT projeté: (400, 384)
Variance expliquée ViT: 1.000


In [11]:
# Tester différentes valeurs de α
alphas = [0.3, 0.5, 0.7]
results_s2 = {}

for alpha in alphas:
    # Moyenne pondérée
    multimodal = alpha * text_proj + (1 - alpha) * vit_proj
    
    # Similarité et top-5
    similarity = cosine_similarity(multimodal)
    
    results = []
    for anchor_id in anchor_ids:
        anchor_idx = df[df['id'] == anchor_id].index[0]
        similarities = similarity[anchor_idx]
        similarities[anchor_idx] = -1
        top5_indices = np.argsort(similarities)[-5:][::-1]
        top5_ids = df.iloc[top5_indices]['id'].values
        
        results.append({
            'anchor_id': anchor_id,
            'top5_ids': list(top5_ids)
        })
    
    df_alpha = pd.DataFrame(results)
    metrics = evaluate_method(df_alpha, ground_truth)
    results_s2[alpha] = metrics
    
    print(f"α={alpha:.1f}: P@5={metrics['precision']:.3f}, R@5={metrics['recall']:.3f}, MRR={metrics['mrr']:.3f}")

α=0.3: P@5=0.473, R@5=0.473, MRR=0.741
α=0.5: P@5=0.500, R@5=0.500, MRR=0.833
α=0.7: P@5=0.447, R@5=0.447, MRR=0.739


In [12]:
# Trouver le meilleur α
best_alpha = max(results_s2.keys(), key=lambda k: results_s2[k]['precision'])
best_metrics_s2 = results_s2[best_alpha]

print(f"\nMeilleur α: {best_alpha}")
print(f"Stratégie 2 (α={best_alpha}): P@5={best_metrics_s2['precision']:.3f}, R@5={best_metrics_s2['recall']:.3f}, MRR={best_metrics_s2['mrr']:.3f}")

# Sauvegarder les résultats du meilleur α
multimodal_s2 = best_alpha * text_proj + (1 - best_alpha) * vit_proj
np.save('results/strategy2_multimodal_embeddings.npy', multimodal_s2)

# Régénérer le top-5 avec le meilleur α
similarity_s2 = cosine_similarity(multimodal_s2)
results_s2_final = []
for anchor_id in anchor_ids:
    anchor_idx = df[df['id'] == anchor_id].index[0]
    similarities = similarity_s2[anchor_idx]
    similarities[anchor_idx] = -1
    top5_indices = np.argsort(similarities)[-5:][::-1]
    top5_ids = df.iloc[top5_indices]['id'].values
    
    results_s2_final.append({
        'anchor_id': anchor_id,
        'top5_ids': list(top5_ids)
    })

df_strategy2 = pd.DataFrame(results_s2_final)
df_strategy2.to_csv('results/strategy2_weighted_top5.csv', index=False)
print("\nSauvegardé: results/strategy2_weighted_top5.csv")


Meilleur α: 0.5
Stratégie 2 (α=0.5): P@5=0.500, R@5=0.500, MRR=0.833

Sauvegardé: results/strategy2_weighted_top5.csv


## Stratégie 3 : CLIP (Contrastive Language-Image Pre-training)

**Formule** : Utiliser `openai/clip-vit-base-patch32`

- Encode texte et image dans un **espace commun** (512 dim)
- Les embeddings sont directement comparables par similarité cosinus
- Le modèle a été entraîné pour aligner les deux modalités

In [13]:
import clip
import torch

# Charger CLIP (ViT-B/32 = clip-vit-base-patch32)
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

print(f"Modèle CLIP chargé : ViT-B/32")
print(f"Dimension espace commun: 512")


Modèle CLIP chargé : ViT-B/32
Dimension espace commun: 512


In [14]:
from PIL import Image

# Extraire embeddings CLIP
def get_clip_embeddings(df, image_folder):
    clip_text_embeddings = []
    clip_image_embeddings = []
    
    for idx, row in df.iterrows():
        # Texte
        text_tokens = clip.tokenize([str(row['description'])], truncate=True)
        text_feat = clip_model.encode_text(text_tokens)
        clip_text_embeddings.append(text_feat.detach().cpu().numpy().squeeze())
        
        # Image
        img_path = f"{image_folder}/{row['id']}.jpg"
        try:
            image = Image.open(img_path).convert('RGB')
            image_input = preprocess(image).unsqueeze(0).to(device)
            img_feat = clip_model.encode_image(image_input)
            clip_image_embeddings.append(img_feat.detach().cpu().numpy().squeeze())
        except:
            clip_image_embeddings.append(np.zeros(512))
        
        if (idx + 1) % 50 == 0:
            print(f"  {idx + 1}/{len(df)} produits traités")
    
    return np.array(clip_text_embeddings), np.array(clip_image_embeddings)

clip_text, clip_image = get_clip_embeddings(df, 'data/subset_fashion_dataset/images')
print(f"\nCLIP texte: {clip_text.shape}")
print(f"CLIP image: {clip_image.shape}")


  50/400 produits traités
  100/400 produits traités
  150/400 produits traités
  200/400 produits traités
  250/400 produits traités
  300/400 produits traités
  350/400 produits traités
  400/400 produits traités

CLIP texte: (400, 512)
CLIP image: (400, 512)


In [15]:
# Approche CLIP : moyenne des embeddings texte et image (déjà dans le même espace)
clip_multimodal = (clip_text + clip_image) / 2

# Similarité et top-5
clip_similarity = cosine_similarity(clip_multimodal)

results_s3 = []
for anchor_id in anchor_ids:
    anchor_idx = df[df['id'] == anchor_id].index[0]
    similarities = clip_similarity[anchor_idx]
    similarities[anchor_idx] = -1
    top5_indices = np.argsort(similarities)[-5:][::-1]
    top5_ids = df.iloc[top5_indices]['id'].values
    
    results_s3.append({
        'anchor_id': anchor_id,
        'top5_ids': list(top5_ids)
    })

df_strategy3 = pd.DataFrame(results_s3)
metrics_s3 = evaluate_method(df_strategy3, ground_truth)

print(f"Stratégie 3 (CLIP): P@5={metrics_s3['precision']:.3f}, R@5={metrics_s3['recall']:.3f}, MRR={metrics_s3['mrr']:.3f}")

Stratégie 3 (CLIP): P@5=0.500, R@5=0.500, MRR=0.819


In [16]:
# Sauvegarder stratégie 3
df_strategy3.to_csv('results/strategy3_clip_top5.csv', index=False)
np.save('results/strategy3_multimodal_embeddings.npy', clip_multimodal)
np.save('results/clip_text_embeddings.npy', clip_text)
np.save('results/clip_image_embeddings.npy', clip_image)

print("Sauvegardé:")
print("  - results/strategy3_clip_top5.csv")
print("  - results/strategy3_multimodal_embeddings.npy")
print("  - results/clip_text_embeddings.npy")
print("  - results/clip_image_embeddings.npy")

Sauvegardé:
  - results/strategy3_clip_top5.csv
  - results/strategy3_multimodal_embeddings.npy
  - results/clip_text_embeddings.npy
  - results/clip_image_embeddings.npy


In [17]:
# Comparaison complète avec méthodes unimodales
df_all = pd.DataFrame([
    {'Méthode': 'TF-IDF', 'Type': 'Unimodal Texte', **df_unimodal.iloc[0][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'Embeddings', 'Type': 'Unimodal Texte', **df_unimodal.iloc[1][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'ResNet-50', 'Type': 'Unimodal Visuel', **df_unimodal.iloc[2][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'ViT', 'Type': 'Unimodal Visuel', **df_unimodal.iloc[3][['precision', 'recall', 'mrr']].to_dict()},
    {'Méthode': 'S1: Concaténation', 'Type': 'Multimodal', **metrics_s1},
    {'Méthode': f'S2: Moyenne α={best_alpha}', 'Type': 'Multimodal', **best_metrics_s2},
    {'Méthode': 'S3: CLIP', 'Type': 'Multimodal', **metrics_s3}
])

print("\n" + "="*100)
print("COMPARAISON COMPLÈTE UNIMODAL vs MULTIMODAL")
print("="*100)
print(f"{'Méthode':<25} | {'Type':<15} | {'P@5':>10} | {'R@5':>10} | {'MRR':>10}")
print("-"*100)
for _, row in df_all.iterrows():
    p = f"{row['precision']*100:.1f}%"
    r = f"{row['recall']*100:.1f}%"
    m = f"{row['mrr']:.3f}"
    marker = " ★" if row['precision'] == df_all['precision'].max() else ""
    print(f"{row['Méthode']:<25} | {row['Type']:<15} | {p:>10} | {r:>10} | {m:>10}{marker}")
print("="*100)

df_all.to_csv('results/fusion_strategies_comparison.csv', index=False)
print("\nSauvegardé: results/fusion_strategies_comparison.csv")


COMPARAISON COMPLÈTE UNIMODAL vs MULTIMODAL
Méthode                   | Type            |        P@5 |        R@5 |        MRR
----------------------------------------------------------------------------------------------------
TF-IDF                    | Unimodal Texte  |      36.0% |      36.0% |      0.662
Embeddings                | Unimodal Texte  |      36.7% |      36.7% |      0.661
ResNet-50                 | Unimodal Visuel |      38.7% |      38.7% |      0.364
ViT                       | Unimodal Visuel |      44.7% |      44.7% |      0.316
S1: Concaténation         | Multimodal      |      53.3% |      53.3% |      0.819 ★
S2: Moyenne α=0.5         | Multimodal      |      50.0% |      50.0% |      0.833
S3: CLIP                  | Multimodal      |      50.0% |      50.0% |      0.819

Sauvegardé: results/fusion_strategies_comparison.csv


## Comparaison Finale des 3 Stratégies

In [18]:
# Tableau comparatif des 3 stratégies
df_strategies = pd.DataFrame([
    {'Stratégie': '1. Concaténation', 'Détails': '1152 dim', **metrics_s1},
    {'Stratégie': f'2. Moyenne (α={best_alpha})', 'Détails': '512 dim (PCA)', **best_metrics_s2},
    {'Stratégie': '3. CLIP', 'Détails': '512 dim (espace commun)', **metrics_s3}
])

print("\n" + "="*90)
print("COMPARAISON DES 3 STRATÉGIES DE FUSION")
print("="*90)
print(f"{'Stratégie':<30} | {'Détails':<20} | {'P@5':>10} | {'R@5':>10} | {'MRR':>10}")
print("-"*90)
for _, row in df_strategies.iterrows():
    p = f"{row['precision']*100:.1f}%"
    r = f"{row['recall']*100:.1f}%"
    m = f"{row['mrr']:.3f}"
    print(f"{row['Stratégie']:<30} | {row['Détails']:<20} | {p:>10} | {r:>10} | {m:>10}")
print("="*90)

# Meilleure stratégie
best_strategy = df_strategies.loc[df_strategies['precision'].idxmax()]
print(f"\n★ Meilleure stratégie: {best_strategy['Stratégie']}")
print(f"  P@5: {best_strategy['precision']*100:.1f}%, MRR: {best_strategy['mrr']:.3f}")


COMPARAISON DES 3 STRATÉGIES DE FUSION
Stratégie                      | Détails              |        P@5 |        R@5 |        MRR
------------------------------------------------------------------------------------------
1. Concaténation               | 1152 dim             |      53.3% |      53.3% |      0.819
2. Moyenne (α=0.5)             | 512 dim (PCA)        |      50.0% |      50.0% |      0.833
3. CLIP                        | 512 dim (espace commun) |      50.0% |      50.0% |      0.819

★ Meilleure stratégie: 1. Concaténation
  P@5: 53.3%, MRR: 0.819
